#Why can schema inference be risky for customer and product identifiers?
It is risky as it leads to data truncation,affects table to table relationships and data quality.
Also might corrupt data as well.

#What do StructType and StructField represent?
StructType - Complete collection of fields,represents the entire schema. Contains multiple structField objects
StructField - individual field inside a structType.defines dataType,column name,nullabilty

In [0]:

#What does nullable mean in a Spark schema?
whether a column can contain null values or not.boolean flag(true or false)
#Which type should represent quantity, and why?
integer or decimal
#Why use a decimal type for retail prices?
prevents rouding errors and ensures precision.
#How do DateType and TimestampType differ?
dateType - stores calender date, no time component here
TimestampType - includes time component as well,hours,mins,sec
#What does the CSV header option control?
whether the first line of the CSV file contains column names or not.
#What is the difference between a blank field and a quoted empty string
quoted empty string - a deliberate, present value which contains zero characters
blank field - data is missing or unknown
#What does PERMISSIVE parsing attempt to preserve
sparks default JSON/CSV parsing mode wherein it attempts to preserve all data, even if it is malformed.
#Why do file streams normally require an explicit schema
for consistency across batches(all files),performance and resiliance during failures



In [0]:
data=spark.read.csv("/Volumes/practice/practice_schema/de_test/customers.csv",header=True,inferSchema=True)
display(data)
data.printSchema()

customer_id,name,email,city,signup_date,updated_at
C001,Synthetic Customer 1,buyer1@example.test,mumbai,2026-06-01,2026-06-01T00:00:00.000Z
C002,Synthetic Customer 2,buyer2@example.test,Bengaluru,2026-06-01,2026-06-01T00:00:00.000Z
C003,Synthetic Customer 3,buyer3@example.test,CHENNAI,2026-06-01,2026-06-01T00:00:00.000Z
C004,Synthetic Customer 4,buyer4@example.test,Delhi,2026-06-01,2026-06-01T00:00:00.000Z
C005,Synthetic Customer 5,buyer5@example.test,mumbai,2026-06-01,2026-06-01T00:00:00.000Z
C006,Synthetic Customer 6,buyer6@example.test,Bengaluru,2026-06-01,2026-06-01T00:00:00.000Z
C007,Synthetic Customer 7,buyer7@example.test,CHENNAI,2026-06-01,2026-06-01T00:00:00.000Z
C008,Synthetic Customer 8,buyer8@example.test,Delhi,2026-06-01,2026-06-01T00:00:00.000Z
C009,Synthetic Customer 9,buyer9@example.test,mumbai,2026-06-01,2026-06-01T00:00:00.000Z
C010,Synthetic Customer 10,buyer10@example.test,Bengaluru,2026-06-01,2026-06-01T00:00:00.000Z


root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- updated_at: timestamp (nullable = true)



In [0]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType,
     DateType, TimestampType
)

customers_schema = StructType([
    StructField("customer_id", IntegerType(), False),    
    StructField("name", StringType(), False),
    StructField("email", StringType(), True),
    StructField("city", StringType(), True),
    StructField("signup_date", DateType(), True),
    StructField("updated_at", TimestampType(), True)
   ])

In [0]:
data=spark.read.csv("/Volumes/practice/practice_schema/de_test/order_items.csv",header=True,inferSchema=True)
display(data)
data.printSchema()

line_id,order_id,product_id,quantity,unit_price,discount_pct
L0001_1,O0001,P008,2,700,0.0
L0001_2,O0001,P004,5,400,0.0
L0001_3,O0001,P019,4,1525,0.0
L0002_1,O0002,P008,5,700,0.1
L0002_2,O0002,P001,5,175,0.0
L0003_1,O0003,P015,5,1225,0.05
L0003_2,O0003,P001,2,175,0.1
L0004_1,O0004,P005,2,475,0.05
L0004_2,O0004,P004,1,400,0.05
L0004_3,O0004,P004,3,400,0.05


root
 |-- line_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_pct: double (nullable = true)



In [0]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType,
    DoubleType,  TimestampType
)
from pyspark.sql.functions import to_utc_timestamp

order_items_schema = StructType([
    StructField("line_id", StringType(), True),    
    StructField("order_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", IntegerType(), True),
    StructField("discount_pct", DoubleType (), True)
   ])

#Read a CSV fixture containing a quoted comma in a customer name.
Spark sees the comma inside the quotes as part of the field, not a field separator.Spark reader is 4180 compliant


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.functions import *

schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("order_status", StringType(), True),
    StructField("_corrupt_record", StringType(), True),
])

data = """order_id,customer_id,amount,order_status
1,501,250.00,COMPLETED
2,502,notanumber,PENDING
3,503,120.50,COMPLETED
abc,504,75.00,CANCELLED
5,505,,PENDING"""

dbutils.fs.put("/Volumes/practice/practice_schema/de_test/malformed_orders.csv", data, overwrite=True)

# <-- the missing step: actually read the file as a DataFrame
df = (
    spark.read
    .schema(schema)
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .csv("/Volumes/practice/practice_schema/de_test/malformed_orders.csv")
)

df.show(truncate=False)
df.filter(df["_corrupt_record"].isNotNull()).show(truncate=False)

Wrote 150 bytes.
+--------+-----------+------+------------+------------------------+
|order_id|customer_id|amount|order_status|_corrupt_record         |
+--------+-----------+------+------------+------------------------+
|1       |501        |250.0 |COMPLETED   |NULL                    |
|2       |502        |NULL  |PENDING     |2,502,notanumber,PENDING|
|3       |503        |120.5 |COMPLETED   |NULL                    |
|NULL    |504        |75.0  |CANCELLED   |abc,504,75.00,CANCELLED |
|5       |505        |NULL  |PENDING     |NULL                    |
+--------+-----------+------+------------+------------------------+

+--------+-----------+------+------------+------------------------+
|order_id|customer_id|amount|order_status|_corrupt_record         |
+--------+-----------+------+------------+------------------------+
|2       |502        |NULL  |PENDING     |2,502,notanumber,PENDING|
|NULL    |504        |75.0  |CANCELLED   |abc,504,75.00,CANCELLED |
+--------+-----------+------+-

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("order_status", StringType(), True),
])

path = "/Volumes/practice/practice_schema/de_test/malformed_orders.csv"

# --- DROPMALFORMED ---
df_drop = spark.read.schema(schema).option("header", True).option("mode", "DROPMALFORMED").csv(path)
df_drop.show(truncate=False)  

# --- FAILFAST ---
df_fail = spark.read.schema(schema).option("header", True).option("mode", "FAILFAST").csv(path)
df_fail.show(truncate=False)   

+--------+-----------+------+------------+
|order_id|customer_id|amount|order_status|
+--------+-----------+------+------------+
|1       |501        |250.0 |COMPLETED   |
|3       |503        |120.5 |COMPLETED   |
|5       |505        |NULL  |PENDING     |
+--------+-----------+------+------------+



---------------------------------------------------------------------------
SparkException                            Traceback (most recent call last)
File <command-8478637920487357>, line 18
     16 # --- FAILFAST ---
     17 df_fail = spark.read.schema(schema).option("header", True).option("mode", "FAILFAST").csv(path)
---> 18 df_fail.show(truncate=False)   

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1297, in DataFrame.show(self, n, truncate, vertical)
   1296 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1297     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:941, in DataFrame._show_string(self, n, truncate, vertical)
    923     except ValueError:
    924         raise PySparkTypeError(
    925             errorClass="NOT_EXPECTED_TYPE",
    926             messageParameters={
   (...)    930       

In [0]:
from pyspark.sql.functions import col, current_timestamp

df = (
    df
    .withColumn("source_filename", col("_metadata.file_path"))
    .withColumn("ingestion_timestamp", current_timestamp())
)

df.select("order_id","customer_id","amount","order_status","source_filename", "ingestion_timestamp").show(truncate=False)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8478637920487358>, line 9
      1 from pyspark.sql.functions import col, current_timestamp
      3 df = (
      4     df
      5     .withColumn("source_filename", col("_metadata.file_path"))
      6     .withColumn("ingestion_timestamp", current_timestamp())
      7 )
----> 9 df.select("order_id","customer_id","amount","order_status","source_filename", "ingestion_timestamp").show(truncate=False)

File <command-8478637920487358>, line 4
      1 from pyspark.sql.functions import col, current_timestamp
      3 df = (
----> 4     df
      5     .withColumn("source_filename", col("_metadata.file_path"))
      6     .withColumn("ingestion_timestamp", current_timestamp())
      7 )
      9 df.select("order_id","customer_id","amount","order_status","source_filename", "ingestion_timestamp").show(truncate=False)

File <command-8478